In [2]:
from pathlib import Path
import pandas as pd

from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent

with open(PROJECT_ROOT / "params.yaml", "r") as f:
    params = yaml.safe_load(f)

print("Configuration loaded successfully!")
print(params)

PROJECT_ROOT = Path.cwd().parent

train_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/train_clean.csv"
)

later_df = pd.read_csv(
    PROJECT_ROOT / "data/raw/later.csv"
)

print("Training data:", train_df.shape)
print("Later data:", later_df.shape)

print("\nTraining B-Block average price:",
      train_df.loc[train_df["block"] == "B-Block",
                   "menu_price_rs"].mean())

print("\nLater B-Block average price:",
      later_df.loc[later_df["block"] == "B-Block",
                   "menu_price_rs"].mean())

Configuration loaded successfully!
{'data': {'train_path': 'data/raw/train.csv', 'later_path': 'data/raw/later.csv', 'holdout_path': 'data/holdout_features.csv', 'cleaned_train_path': 'data/processed/train_clean.csv'}, 'training': {'test_size': 0.2, 'random_state': 42}, 'models': {'linear_regression': {}, 'random_forest': {'n_estimators': 200, 'random_state': 42}, 'gradient_boosting': {'n_estimators': 100, 'learning_rate': 0.05, 'random_state': 42}}, 'monitoring': {'psi_bins': 10, 'later_train_fraction': 0.7, 'promotion_threshold': 0.12}, 'mlflow': {'experiment_name': 'MP002_Canteen_Demand_Forecasting', 'registered_model_name': 'canteen_demand_model', 'champion_alias': 'champion'}}
Training data: (1000, 11)
Later data: (300, 11)

Training B-Block average price: 75.05128205128206

Later B-Block average price: 69.96923076923076


In [6]:
TARGET = "plates_sold"

FEATURES = [
    "students_on_campus",
    "menu_price_rs",
    "temp_c",
    "exam_days_left",
    "special_dishes",
    "queue_minutes_yesterday",
    "day_type",
    "block"
]

X = train_df[FEATURES]
y = train_df[TARGET]

X_later = later_df[FEATURES]
y_later = later_df[TARGET]

print("Features:", FEATURES)
print("Target:", TARGET)

Features: ['students_on_campus', 'menu_price_rs', 'temp_c', 'exam_days_left', 'special_dishes', 'queue_minutes_yesterday', 'day_type', 'block']
Target: plates_sold


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify feature types
numeric_features = [
    "students_on_campus",
    "menu_price_rs",
    "temp_c",
    "exam_days_left",
    "special_dishes",
    "queue_minutes_yesterday"
]

categorical_features = [
    "day_type",
    "block"
]

# Numeric preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

numeric_features = [
    "students_on_campus", "menu_price_rs", "temp_c",
    "exam_days_left", "special_dishes", "queue_minutes_yesterday"
]

categorical_features = ["day_type", "block"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.05, random_state=42
    )
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    results[name] = mae

    print(f"{name}: MAE = {mae:.2f}")

print("\nBest model:", min(results, key=results.get))

Linear Regression: MAE = 13.68
Random Forest: MAE = 22.93
Gradient Boosting: MAE = 21.62

Best model: Linear Regression


In [9]:
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# Build final V1 pipeline
v1_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LinearRegression())
])

# Train on all cleaned training data
v1_model.fit(X, y)

# Evaluate on later dataset
later_predictions = v1_model.predict(X_later)

v1_later_mae = mean_absolute_error(
    y_later,
    later_predictions
)

print("V1 Model: Linear Regression")
print("MAE on later dataset:", round(v1_later_mae, 2))

V1 Model: Linear Regression
MAE on later dataset: 33.28


In [5]:
later_results = later_df.copy()
later_results["prediction"] = later_predictions
later_results["absolute_error"] = abs(
    later_results["plates_sold"] - later_results["prediction"]
)

print(
    later_results.groupby("block")["absolute_error"]
    .mean()
    .round(2)
)

block
A-Block    24.60
B-Block    50.72
Hostel     23.61
Name: absolute_error, dtype: float64


In [6]:
import joblib

MODEL_PATH = PROJECT_ROOT / "models"
MODEL_PATH.mkdir(exist_ok=True)

joblib.dump(v1_model, MODEL_PATH / "v1_model.pkl")

print("V1 model saved successfully!")

V1 model saved successfully!
